In [1]:
from copy import deepcopy

In [2]:
import sys 

sys.path.append("../")

In [3]:
from src import ASTPruner, ASTTreeOperator, SqlglotOperator, OntologyOperator, TreeNode

In [40]:
query = """SELECT l.loan_id, l.status FROM Loan l WHERE l.account_id IN (SELECT account_id FROM Account WHERE district_id = 'd002');"""

In [71]:
from apted import APTED, Config

query = """SELECT\n  amount,\n  bank_to\nFROM \"order\"\nWHERE\n  account_id = 1 AND account_to = 87144583 AND k_symbol = 'SIPO'"""
sql_op = SqlglotOperator(query)
tree_op = ASTTreeOperator(sql_op)
ast_1 = tree_op.root

query2 = """SELECT t1.\"amount\", t1.\"bank_to\"\nFROM \"order\" AS t1\nWHERE t1.\"account_id\" IS NOT NULL AND t1.\"account_id\" >= 0 AND t1.\"account_to\" IS NOT NULL AND t1.\"k_symbol\" IS NOT NULL AND t1.\"k_symbol\" >= 0"""
sql_op2 = SqlglotOperator(query2)
tree_op2 = ASTTreeOperator(sql_op2)
ast_2 = tree_op2.root


In [73]:
print(query)

SELECT
  amount,
  bank_to
FROM "order"
WHERE
  account_id = 1 AND account_to = 87144583 AND k_symbol = 'SIPO'


In [74]:
print(query2)

SELECT t1."amount", t1."bank_to"
FROM "order" AS t1
WHERE t1."account_id" IS NOT NULL AND t1."account_id" >= 0 AND t1."account_to" IS NOT NULL AND t1."k_symbol" IS NOT NULL AND t1."k_symbol" >= 0


In [72]:

class ASTConfig(Config):
    # def delete(self, node):
    #     return 1 
    
    # def insert(self, node):
    #     return 1 
    
    def rename(self, node1, node2):
        if node1.name == node2.name:
            if node1.name == "ColumnRef":
                return 0 if node1.refcol == node2.refcol else 1
            elif node1.name == "TableRef":
                return 0 if node1.reftable == node2.reftable else 1
            elif node1.name == "Literal":
                return 0 if node1.value == node2.value else 1
            else:
                return 0 if node1.name == node2.name else 1
        else:
            return 1
        # return 0 if (node1.name == node2.name) and (node1.kind == node2.kind) else 1
    
    def children(self, node):
        return super().children(node)
    

apted = APTED(ast_1, ast_2, config=ASTConfig())
ted = apted.compute_edit_distance()
mapping = apted.compute_edit_mapping()
print("distance:", ted, "mapping:", mapping)
# ast_2.print_tree()

distance: 15 mapping: [(<(se10dd8) Statement |  (SelectStatement) [remove=False]>, <(s3906d9) Statement |  (SelectStatement) [remove=False]>), (<(ne10dd8) Clause |  (SelectClause) [remove=False]>, <(n3906d9) Clause |  (SelectClause) [remove=False]>), (<(n1d86a9) Expression |  (ColumnRef) [reftable=None, refcol=amount] [remove=False]>, <(nab48e8) Expression |  (ColumnRef) [reftable=t1, refcol=amount] [remove=False]>), (<(ndee402) Expression |  (ColumnRef) [reftable=None, refcol=bank_to] [remove=False]>, <(n0be575) Expression |  (ColumnRef) [reftable=t1, refcol=bank_to] [remove=False]>), (<(n408f1c) Clause |  (FromClause) [remove=False]>, <(ne2b3ca) Clause |  (FromClause) [remove=False]>), (<(nd97ccf) Expression |  (TableRef) [reftable=order] [remove=False]>, <(nd79cf3) Expression |  (TableRef) [reftable=order] [remove=False] [refalias=t1]>), (<(n4c2385) Clause |  (WhereClause) [remove=False] [value=account_id = 1 AND account_to = 87144583 AND k_symbol = 'SIPO']>, <(nf2b905) Clause |  (W

In [5]:
# query = "WITH AccountTransactions AS (SELECT a.account_id, a.frequency, t.type FROM Account a JOIN Transaction t ON a.account_id = t.account_id) SELECT * FROM AccountTransactions WHERE type = 'withdrawal';"

In [43]:
query = """
SELECT
    c.duration,
    c.date,
    (
        SELECT COUNT(*)
        FROM Order o
        WHERE o.account_id = c.account_id
    ) AS total_orders,
    (
        SELECT AVG(o2.amount)
        FROM Order o2
        WHERE o2.account_id = c.account_id
          AND o2.amount > 100
    ) AS avg_recent_order_amount
FROM Loan c
WHERE c.status = 'active';
"""

In [44]:
# query = """INSERT INTO "Loan" (loan_id, account_id, amount) VALUES ('l123', 'a456', 5000);"""

In [8]:
onto_path = "../data/ontology_file/argos_v2.0.rdf"
finance_path = "../experiment/outputs/agenta_financial_full_v8_20260215T205141Z_db_abox/financial_abox.rdf"
agent_id = "a0012"

# 1. Create an instance of the class
instantiator = OntologyOperator(onto_path, finance_path)

table entities: {'card': 't002', 'trans': 't008', 'disp': 't004', 'district': 't005', 'client': 't003', 'order': 't007', 'xxxxx': 't00x', 'loan': 't006', 'account': 't001'}
column lookup: {('card', 'type'): 'c007', ('trans', 'type'): 'c007', ('disp', 'type'): 'c007', ('district', 'A6'): 'c016', ('client', 'gender'): 'c010', ('trans', 'account'): 'c040', ('district', 'A2'): 'c012', ('district', 'A4'): 'c014', ('order', 'order_id'): 'c032', ('order', 'bank_to'): 'c033', ('trans', 'bank'): 'c039', ('trans', 'account_id'): 'c001', ('disp', 'account_id'): 'c001', ('order', 'account_id'): 'c001', ('loan', 'account_id'): 'c001', ('account', 'account_id'): 'c001', ('disp', 'client_id'): 'c009', ('client', 'client_id'): 'c009', ('district', 'district_id'): 'c002', ('client', 'district_id'): 'c002', ('account', 'district_id'): 'c002', ('client', 'birth_date'): 'c011', ('trans', 'date'): 'c004', ('loan', 'date'): 'c004', ('account', 'date'): 'c004', ('trans', 'k_symbol'): 'c035', ('order', 'k_sym

In [47]:
query = """
WITH ClientAccounts AS (SELECT c.client_id, a.account_id FROM "Client" c JOIN "Disposition" d ON c.client_id = d.client_id JOIN "Account" a ON d.account_id = a.account_id) SELECT * FROM ClientAccounts;
 """

In [48]:
query = """
SELECT account, COUNT(type) AS types FROM Transaction WHERE account_id IN (SELECT account_id FROM Account WHERE district_id = 'd002');
"""


In [49]:
query = """
SELECT l.duration, COUNT(l.loan_id) as loan_count FROM "Client" c LEFT JOIN "Loan" l ON c.client_id = l.account_id GROUP BY c.client_id;
"""

In [6]:
query = """SELECT\n  amount,\n  balance,\n  date,\n  type\nFROM trans\nWHERE\n  NOT account IS NULL AND NOT amount IS NULL AND NOT trans_id IS NULL"""

In [ ]:
# query = """
# SELECT l.* FROM "Loan" l WHERE l.account_id IN (SELECT a.account_id FROM "Account" a WHERE a.district_id = 1);
# """

sql_op = SqlglotOperator(query)
tree_op = ASTTreeOperator(sql_op)
# tree_op.pretty_print()
instantiator.instantiate_ontology(tree_op, agent_id)
tree_op.pretty_print()
# 3. Run the reasoner and save the ontology
# instantiator.reason_and_save(output_path, save=True)

└── <(sa2d311) Statement |  (SelectStatement) [remove=False]>
    ├── <(na2d311) Clause |  (SelectClause) [remove=False]>
    │   ├── <(nda4eb1) Expression |  (ColumnRef) [reftable=None, refcol=amount] [remove=False]>
    │   ├── <(n1a3d5c) Expression |  (ColumnRef) [reftable=None, refcol=balance] [remove=False]>
    │   ├── <(n7706ce) Expression |  (ColumnRef) [reftable=None, refcol=date] [remove=False]>
    │   └── <(n72a955) Expression |  (ColumnRef) [reftable=None, refcol=type] [remove=False]>
    ├── <(n2deacd) Clause |  (FromClause) [remove=False]>
    │   └── <(n391e43) Expression |  (TableRef) [reftable=trans] [remove=False]>
    └── <(n2bf1dc) Clause |  (WhereClause) [remove=False] [value=NOT account IS NULL AND NOT amount IS NULL AND NOT trans_id IS NULL]>
        └── <(nf9b88a) Expression |  (Operator) [remove=False] [logics=True] [value=NOT account IS NULL AND NOT amount IS NULL]>
            ├── <(n507a7c) Expression |  (Operator) [remove=False] [logics=True] [value=NOT ac

In [51]:
status_set = instantiator.get_column_ref_instances() | instantiator.get_table_ref_instances()
print("Rule set: ", ", ".join(set([s.get("Rule") for s in status_set.values() if s.get("Rule") is not None])))
print("Policy set: ", ", ".join(set([s.get("Policy") for s in status_set.values() if s.get("Policy") is not None])))

pruner = ASTPruner(instantiator)
# 4. Prune the AST based on the ontology state
pruner.prune()
print(sql_op.to_sql())

Rule set:  
Policy set:  
--- Starting AST Pruning Process ---
Query before pruning: SELECT l.duration, COUNT(l.loan_id) AS loan_count FROM "Client" AS c LEFT JOIN "Loan" AS l ON c.client_id = l.account_id GROUP BY c.client_id
--> Stage 1: Pruning based on table statuses...
--> Stage 2: Pruning based on column statuses...
     - Initial violated column IDs: []
     - No violated columns found.
SELECT
  l.duration,
  COUNT(l.loan_id) AS loan_count
FROM "Client" AS c
LEFT JOIN "Loan" AS l
  ON c.client_id = l.account_id
GROUP BY
  c.client_id


---

In [65]:
query = """
SELECT
    c.duration,
    c.date,
    (
        SELECT COUNT(*)
        FROM Order o
        WHERE o.account_id = c.account_id
    ) AS total_orders,
    (
        SELECT AVG(o2.amount)
        FROM Order o2
        WHERE o2.account_id = c.account_id
          AND o2.amount > 100
    ) AS avg_recent_order_amount
FROM Loan c
WHERE c.status = 'active';"""

In [66]:
sql_op = SqlglotOperator(query)
# view the sqlglot AST
sql_op.ast

Select(
  expressions=[
    Column(
      this=Identifier(this=duration, quoted=False),
      table=Identifier(this=c, quoted=False)),
    Column(
      this=Identifier(this=date, quoted=False),
      table=Identifier(this=c, quoted=False)),
    Alias(
      this=Subquery(
        this=Select(
          expressions=[
            Count(
              this=Star(),
              big_int=True)],
          from=From(
            this=Table(
              this=Identifier(this=Order, quoted=False),
              alias=TableAlias(
                this=Identifier(this=o, quoted=False)))),
          where=Where(
            this=EQ(
              this=Column(
                this=Identifier(this=account_id, quoted=False),
                table=Identifier(this=o, quoted=False)),
              expression=Column(
                this=Identifier(this=account_id, quoted=False),
                table=Identifier(this=c, quoted=False)))))),
      alias=Identifier(this=total_orders, quoted=False)),
    A

In [67]:
tree_op = ASTTreeOperator(sql_op)
print("\nCustom AST Tree:")
tree_op.pretty_print()


Custom AST Tree:
└── <(s7e8a7c) Statement |  (SelectStatement) [remove=False]>
    ├── <(n7e8a7c) Clause |  (SelectClause) [remove=False]>
    │   ├── <(nd87471) Expression |  (ColumnRef) [reftable=c, refcol=duration] [remove=False]>
    │   ├── <(n5690c7) Expression |  (ColumnRef) [reftable=c, refcol=date] [remove=False]>
    │   ├── <(nf0128f) Expression |  (Alias) [alias=total_orders] [remove=False]>
    │   │   └── <(n1c0ce0) Expression |  (Subquery) [remove=False]>
    │   │       ├── <(n4c410f) Clause |  (SelectClause) [remove=False]>
    │   │       │   └── <(n046bc9) Expression |  (Function) [remove=False]>
    │   │       │       └── <(nb18503) Expression |  (Wildcard) [remove=False]>
    │   │       ├── <(n965eba) Clause |  (FromClause) [remove=False]>
    │   │       │   └── <(n3d6059) Expression |  (TableRef) [reftable=Order] [remove=False] [refalias=o]>
    │   │       └── <(nde8324) Clause |  (WhereClause) [remove=False] [value=o.account_id = c.account_id]>
    │   │    

In [68]:
onto_path = "../data/ontology_file/aputv5e.rdf"
output_path = "../data/ontology_file/resolved_query_instances.rdf"
agent_id = "a0012"

# 1. Create an instance of the class
instantiator = OntologyOperator(onto_path)

instantiator.instantiate_ontology(tree_op, agent_id)
tree_op.pretty_print()
# 3. Run the reasoner and save the ontology
# instantiator.reason_and_save(output_path, save=True)

table entities: {'Card': 't007', 'Disposition': 't005', 'District': 't008', 'Order': 't001', 'Transaction': 't002', 'Account': 't003', 'Loan': 't004', 'Client': 't006', 'xxxxx': 't00x'}
column lookup: {('xxxxx', 'ooooo'): 'c00x', ('Card', 'type'): 'c026', ('Order', 'order_id'): 'c001', ('Order', 'account_to'): 'c004', ('Order', 'k_symbol'): 'c006', ('Transaction', 'trans_id'): 'c007', ('Loan', 'loan_id'): 'c016', ('Card', 'disp_id'): 'c020', ('Disposition', 'disp_id'): 'c020', ('Disposition', 'client_id'): 'c021', ('Client', 'client_id'): 'c021', ('Card', 'card_id'): 'c025', ('Transaction', 'k_symbol'): 'c032', ('Disposition', 'account_id'): 'c002', ('Order', 'account_id'): 'c002', ('Transaction', 'account_id'): 'c002', ('Account', 'account_id'): 'c002', ('Loan', 'account_id'): 'c002', ('Client', 'account_id'): 'c002', ('Order', 'amount'): 'c005', ('Transaction', 'date'): 'c008', ('Transaction', 'type'): 'c009', ('District', 'district_id'): 'c014', ('Account', 'district_id'): 'c014', (

In [69]:
print("Column reference node status: ", instantiator.get_column_ref_instances())
print("Table reference node status: ", instantiator.get_table_ref_instances())

Column reference node status:  {'nd87471': {'Status': 'Aligned'}, 'n5690c7': {'Status': 'Aligned'}, 'n50e2e7': {'Status': 'Aligned'}, 'n7b101e': {'Status': 'Aligned'}, 'n52194a': {'Status': 'Aligned'}, 'n57a5c5': {'Status': 'Aligned'}, 'nc06038': {'Status': 'Aligned'}, 'n0790be': {'Status': 'Aligned'}, 'n1c3cbc': {'Status': 'Aligned'}}
Table reference node status:  {'n3d6059': {'Status': 'Aligned'}, 'n155b6e': {'Status': 'Aligned'}, 'nc4abb7': {'Status': 'Aligned'}}


In [70]:
pruner = ASTPruner(instantiator)

# # 4. Prune the AST based on the ontology state
pruner.prune()

print(sql_op.to_sql())

--- Starting AST Pruning Process ---
Query before pruning: SELECT c.duration, c.date, (SELECT COUNT(*) FROM Order AS o WHERE o.account_id = c.account_id) AS total_orders, (SELECT AVG(o2.amount) FROM Order AS o2 WHERE o2.account_id = c.account_id AND o2.amount > 100) AS avg_recent_order_amount FROM Loan AS c WHERE c.status = 'active'
--> Stage 1: Pruning based on table statuses...
--> Stage 2: Pruning based on column statuses...
     - Initial violated column IDs: []
     - No violated columns found.
SELECT
  c.duration,
  c.date,
  (
    SELECT
      COUNT(*)
    FROM Order AS o
    WHERE
      o.account_id = c.account_id
  ) AS total_orders,
  (
    SELECT
      AVG(o2.amount)
    FROM Order AS o2
    WHERE
      o2.account_id = c.account_id AND o2.amount > 100
  ) AS avg_recent_order_amount
FROM Loan AS c
WHERE
  c.status = 'active'


In [64]:
instantiator.cleanup()

--> Cleanup: Attempting to delete 15 instances...
    - Deleted instance: aputv5e.s788ed2
    - Deleted instance: aputv5e.n788ed2
    - Deleted instance: aputv5e.nbf4d67
    - Deleted instance: aputv5e.nb19c2a
    - Deleted instance: aputv5e.nc124d9
    - Deleted instance: aputv5e.nb83096
    - Deleted instance: aputv5e.n695443
    - Deleted instance: aputv5e.n787961
    - Deleted instance: aputv5e.ndcc065
    - Deleted instance: aputv5e.nfa8d85
    - Deleted instance: aputv5e.n91e8aa
    - Deleted instance: aputv5e.n224e09
    - Deleted instance: aputv5e.n909998
    - Deleted instance: aputv5e.na5d834
    - Deleted instance: aputv5e.n93e0da
--> Cleanup complete. Successfully deleted 15 instances.


In [ ]:
import sqlglot
from sqlglot import exp
from sqlglot.expressions import Expression

def sub_condition_exists(full_condition: str, sub_condition: str) -> bool:
    """
    Checks if a sub-condition is structurally present in a full condition
    by parsing both into an Abstract Syntax Tree (AST).

    Args:
        full_condition (str): The complete WHERE clause string.
        sub_condition (str): The sub-condition string to search for.

    Returns:
        True if a structurally identical sub-condition is found, False otherwise.
    """
    try:
        # Parse both conditions into their respective ASTs
        full_ast: Expression = sqlglot.parse_one(full_condition, read="where").this
        sub_ast: Expression = sqlglot.parse_one(sub_condition, read="where").this
    except Exception:
        # If either string is invalid SQL, they cannot match.
        return False

    # Walk the full AST and check if any node is structurally equal
    # to the sub-condition's AST. sqlglot's `==` operator on expressions
    # performs this structural comparison, ignoring whitespace and formatting.
    for node in full_ast.walk():
        if node == sub_ast:
            return True
            
    return False

# --- Example Usage ---
full_clause = "c.status = 'active' AND (l.amount > 5000 OR l.duration = 60)"
sub_clause_to_find = "l.amount>5000"

print(f"Does the sub-condition exist? -> {sub_condition_exists(full_clause, sub_clause_to_find)}")
# Expected Output: Does the sub-condition exist? -> True

Does the sub-condition exist? -> False


In [ ]:
import re

def normalize_condition(condition: str) -> str:
    """
    Cleans up a single SQL condition string to make it comparable.
    - Converts to lowercase
    - Removes extra whitespace
    - Removes whitespace around operators
    """
    # Convert to lowercase and strip leading/trailing whitespace
    norm = condition.strip().lower()
    # Replace multiple spaces with a single space
    norm = re.sub(r'\s+', ' ', norm)
    # Remove whitespace around operators like =, >, <, etc.
    norm = re.sub(r'\s*([=<>!]+)\s*', r'\1', norm)
    return norm

def sub_condition_exists_no_parser(full_condition: str, sub_condition: str) -> bool:
    """
    Checks if a sub-condition exists in a full condition without parsing SQL.

    LIMITATIONS:
    - Only works reliably for conditions joined by AND.
    - Does not understand OR logic or complex nested parentheses.
    - Does not understand SQL semantics (e.g., BETWEEN).
    """
    if not sub_condition:
        return True
    if not full_condition:
        return False

    try:
        # Split the full condition by 'AND' (case-insensitive) and normalize each part
        full_parts = {
            normalize_condition(part)
            for part in re.split(r'\s+AND\s+', full_condition, flags=re.IGNORECASE)
        }

        # Split the sub-condition by 'AND' and normalize each part
        sub_parts = {
            normalize_condition(part)
            for part in re.split(r'\s+AND\s+', sub_condition, flags=re.IGNORECASE)
        }

        # Check if the set of sub-condition parts is a subset of the full condition parts
        return sub_parts.issubset(full_parts)
    except Exception:
        # If any regex or other error occurs, assume failure
        return False

# --- Example Usage ---
full_clause = "c.status = 'active' AND c.age > 30 AND c.gender = 'F'"

# Test 1: Simple case (should be True)
sub1 = "c.age > 30"
print(f"'{sub1}' exists? -> {sub_condition_exists_no_parser(full_clause, sub1)}")

# Test 2: Different order and whitespace (should be True)
sub2 = "c.gender='F'  AND  c.status = 'active'"
print(f"'{sub2}' exists? -> {sub_condition_exists_no_parser(full_clause, sub2)}")

# Test 3: Non-existent condition (should be False)
sub3 = "c.age < 20"
print(f"'{sub3}' exists? -> {sub_condition_exists_no_parser(full_clause, sub3)}")

# Test 4: A case where this method FAILS (involving OR)
full_clause_with_or = "c.status = 'active' OR c.age > 30"
sub4 = "c.age > 30"
print(f"\n--- LIMITATION EXAMPLE ---")
print(f"'{sub4}' in '{full_clause_with_or}'? -> {sub_condition_exists_no_parser(full_clause_with_or, sub4)}")
# This will incorrectly return False because the logic doesn't understand OR.

'c.age > 30' exists? -> True
'c.gender='F'  AND  c.status = 'active'' exists? -> True
'c.age < 20' exists? -> False

--- LIMITATION EXAMPLE ---
'c.age > 30' in 'c.status = 'active' OR c.age > 30'? -> False


In [20]:
from sqlglot import exp

# 1. Manually build the column and subquery nodes (this part is correct)
col_node = exp.Column(this=exp.Identifier(this='district_id', quoted=False))

subquery_node = exp.Subquery(
    this=exp.Select(
        expressions=[
            exp.Column(this=exp.Identifier(this='district_id', quoted=False))
        ],
        from_=exp.From(
            this=exp.Table(this=exp.Identifier(this='District', quoted=True))
        ),
        where=exp.Where(
            this=exp.EQ(
                this=exp.Column(this=exp.Identifier(this='name', quoted=False)),
                expression=exp.Literal(this='Prague', is_string=True)
            )
        )
    )
)

# 2. THE CORRECT WAY TO CREATE THE 'In' NODE:
#    Pass the subquery in a list to the 'expressions' argument.
print("--- Creating the In node correctly using expressions=[...] ---")
in_node = exp.In(this=col_node, expressions=[subquery_node])


# 3. NOW, ACCESSING '.expressions' WILL WORK AS INTENDED:
print(f"\nThe .expressions attribute now contains: {in_node.expressions}")

# The subquery is the first (and only) item in the list.
the_query_section = in_node.expressions[0]

print(f"\nType of the query section: {type(the_query_section)}")
print(f"SQL of the query section: {the_query_section.sql()}")

--- Creating the In node correctly using expressions=[...] ---

The .expressions attribute now contains: [Subquery(
  this=Select(
    expressions=[
      Column(
        this=Identifier(this=district_id, quoted=False))],
    from_=From(
      this=Table(
        this=Identifier(this='District', quoted=True))),
    where=Where(
      this=EQ(
        this=Column(
          this=Identifier(this=name, quoted=False)),
        expression=Literal(this='Prague', is_string=True)))))]

Type of the query section: <class 'sqlglot.expressions.Subquery'>
SQL of the query section: (SELECT district_id WHERE name = 'Prague')


In [20]:
import sqlglot
from sqlglot import exp

# The SQL query as a text string
sql_string = """
SELECT *
FROM "Client"
WHERE district_id IN (
    SELECT district_id
    FROM "District"
    WHERE name = 'Prague'
)
"""

print(f"--- Using sqlglot version: {sqlglot.__version__} ---")

# Let sqlglot parse the string into an AST
try:
    parsed_ast = sqlglot.parse_one(sql_string)
    in_node = parsed_ast.find(exp.In)

    if not in_node:
        print("\nERROR: Could not find an 'In' node in the parsed query.")
    else:
        print("\n--- Full Inspection of the Parsed 'In' Node ---")

        # Use vars() to get the instance dictionary, which shows stored arguments.
        # This is the most important part for debugging.
        print("\n1. Inspecting `vars(in_node)`:")
        try:
            print(vars(in_node))
        except TypeError:
            print("`vars()` not applicable for this object.")

        # Let's try to access some likely candidates if they exist
        print("\n2. Checking potential attributes:")
        potential_attributes = ['this', 'expressions', 'query', 'unnest', 'kind', 'subquery']
        for attr_name in potential_attributes:
            if hasattr(in_node, attr_name):
                value = getattr(in_node, attr_name)
                # To keep the output clean, we'll just show the type for complex objects
                value_repr = type(value) if isinstance(value, exp.Expression) else value
                print(f"  - Attribute '{attr_name}' exists. Value/Type: {value_repr}")
            else:
                print(f"  - Attribute '{attr_name}' does NOT exist.")

except Exception as e:
    print(f"\nAn error occurred during parsing or inspection: {e}")


--- Using sqlglot version: 27.6.0 ---

--- Full Inspection of the Parsed 'In' Node ---

1. Inspecting `vars(in_node)`:
{}

2. Checking potential attributes:
  - Attribute 'this' exists. Value/Type: <class 'sqlglot.expressions.Column'>
  - Attribute 'expressions' exists. Value/Type: []
  - Attribute 'query' does NOT exist.
  - Attribute 'unnest' exists. Value/Type: <bound method Expression.unnest of In(
  this=Column(
    this=Identifier(this=district_id, quoted=False)),
  query=Subquery(
    this=Select(
      expressions=[
        Column(
          this=Identifier(this=district_id, quoted=False))],
      from=From(
        this=Table(
          this=Identifier(this='District', quoted=True))),
      where=Where(
        this=EQ(
          this=Column(
            this=Identifier(this=name, quoted=False)),
          expression=Literal(this='Prague', is_string=True))))))>
  - Attribute 'kind' does NOT exist.
  - Attribute 'subquery' does NOT exist.
